In [54]:
#Flash Attention 2 

In [55]:
import torch
from math import sqrt

Require: Q ∈ R
Nq×d , K, V ∈ R Nk×d
tile sizes Bq, Bk

In [56]:
Nq = 8   # number of query rows
Nk = 8   # number of key rows
d  = 8   # head dimension


Bq = 2   # query block size
Bk = 2   # key block size


Q  = torch.randint(0, 10, (Nq, d)).float()   # queries
K  = torch.randint(0, 10, (Nk, d)).float()   # keys
V  = torch.randint(0, 10, (Nk, d)).float()   # values


O  = torch.zeros(Nq, d)   
L  = torch.zeros(Nq)      

# Gradient of loss w.r.t. output
dO = torch.randint(0, 10, (Nq, d)).float()

print(f"Q, K, V, O, dO : {Q.shape}")
print(f"L              : {L.shape}")
print(f"Tile sizes     : Bq={Bq}, Bk={Bk}")
print(f"Num Q blocks   : {Nq // Bq}")
print(f"Num K blocks   : {Nk // Bk}")


Q, K, V, O, dO : torch.Size([8, 8])
L              : torch.Size([8])
Tile sizes     : Bq=2, Bk=2
Num Q blocks   : 4
Num K blocks   : 4


In [57]:
Q, K, V

(tensor([[9., 2., 2., 8., 0., 0., 3., 8.],
         [6., 0., 5., 3., 7., 7., 7., 3.],
         [3., 6., 9., 0., 4., 7., 9., 4.],
         [3., 5., 3., 6., 1., 0., 9., 0.],
         [7., 7., 9., 3., 3., 7., 6., 6.],
         [7., 1., 6., 9., 5., 3., 4., 9.],
         [9., 8., 3., 0., 7., 1., 2., 0.],
         [7., 8., 2., 5., 7., 1., 5., 0.]]),
 tensor([[5., 8., 8., 8., 2., 3., 7., 5.],
         [0., 3., 1., 0., 3., 9., 1., 9.],
         [8., 7., 6., 2., 6., 6., 0., 3.],
         [8., 0., 7., 0., 7., 9., 1., 6.],
         [2., 8., 0., 3., 3., 8., 6., 1.],
         [1., 4., 7., 6., 6., 2., 2., 2.],
         [8., 7., 8., 0., 7., 6., 1., 0.],
         [6., 3., 7., 4., 7., 7., 9., 4.]]),
 tensor([[5., 1., 7., 7., 6., 2., 1., 2.],
         [1., 3., 4., 4., 6., 5., 4., 9.],
         [5., 0., 3., 4., 6., 4., 2., 5.],
         [4., 7., 1., 5., 3., 1., 7., 6.],
         [1., 9., 1., 8., 5., 0., 7., 1.],
         [8., 9., 5., 8., 5., 0., 9., 7.],
         [5., 8., 9., 7., 2., 5., 4., 2.],
       

Split Q into Tq =⌈Nq/Bq⌉
tiles Q1, . . . , QTq
of size Bq × d

In [58]:
Tq = Nq/Bq
Tk = Nk/Bk
Tq, Tk

(4.0, 4.0)

In [59]:
# unfold(dim, size, step) on dim=0 → (Tq, d, Bq), then transpose to (Tq, Bq, d)
Qis = Q.unfold(0, Bq,Bq).transpose(1, 2)
Kis = K.unfold(0, Bk, Bk).transpose(1,2)
Vis = V.unfold(0, Bk, Bk).transpose(1, 2)

    



In [60]:
Qis.shape, Qis

(torch.Size([4, 2, 8]),
 tensor([[[9., 2., 2., 8., 0., 0., 3., 8.],
          [6., 0., 5., 3., 7., 7., 7., 3.]],
 
         [[3., 6., 9., 0., 4., 7., 9., 4.],
          [3., 5., 3., 6., 1., 0., 9., 0.]],
 
         [[7., 7., 9., 3., 3., 7., 6., 6.],
          [7., 1., 6., 9., 5., 3., 4., 9.]],
 
         [[9., 8., 3., 0., 7., 1., 2., 0.],
          [7., 8., 2., 5., 7., 1., 5., 0.]]]))

So what happenedis is we create 4 blocks of of 2 rows and d col each


In [61]:
Ois = O.unfold(0, Bq, Bq).transpose(1, 2)
Lis = L.unfold(0, Bq, Bq)  
Ois.shape, Lis.shape, Lis,

(torch.Size([4, 2, 8]),
 torch.Size([4, 2]),
 tensor([[0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.]]))

In [62]:
# Online softmax: safe softmax computed incrementally over K blocks
# For each query row, we maintain:
#   mi  = running max of scores (for numerical stability)
#   li  = running sum of exp(scores - mi)
#   Oi  = running weighted sum (unnormalized)
# At the end: O = Oi / li, L = mi + log(li)

In [63]:

Tq_int = int(Tq)
Tk_int = int(Tk)

for i in range(Tq_int):
    print(f"{'='*70}")
    print(f"OUTER LOOP: Q block i={i}  (rows {i*Bq}:{(i+1)*Bq} of Q)")
    print(f"{'='*70}")

    # Load Qi from global memory
    Qi = Qis[i]  # (Bq, d)
    print(f"  Qi shape: {Qi.shape}")
    print(f"  Qi:\n{Qi}\n")

    # Init Oi=0, li=0, mi=-inf (per query row in this block)
    Oi = torch.zeros(Bq, d)
    li = torch.zeros(Bq)
    mi = torch.full((Bq,), -float('inf'))  # max for each row in block
    print(f"  Init → mi: {mi}  |  li: {li}")
    print(f"  Init → Oi shape: {Oi.shape} (all zeros)\n")

    for j in range(Tk_int):
        print(f"  --- Inner loop: K block j={j}  (rows {j*Bk}:{(j+1)*Bk} of K) ---")

        # Load Kj, Vj from global memory
        Kj = Kis[j]  # (Bk, d)
        Vj = Vis[j]  # (Bk, d)
        print(f"    Kj {Kj.shape}:\n{Kj}")
        print(f"    Vj {Vj.shape}:\n{Vj}\n")

        # Compute pre-softmax scores: Sij = Qi @ Kj^T / sqrt(d)  → (Bq, Bk)
        Sij = Qi @ Kj.T
        Sij = Sij / sqrt(d)
        print(f"    Sij = Qi @ Kj^T / sqrt(d)  {Sij.shape}:\n{Sij}")

        # Row-wise max of current scores
        row_max_Sij = Sij.max(dim=1).values
        print(f"    rowmax(Sij): {row_max_Sij}")

        # Compute the new row-wise max
        mi_old = mi.clone()
        mij = torch.max(mi, row_max_Sij)
        print(f"    mi_old: {mi_old}  →  mi_new = max(mi_old, rowmax): {mij}")

        # Adjust scores by new max and exponentiate
        Pij = torch.exp(Sij - mij.unsqueeze(1))
        print(f"    Pij = exp(Sij - mi_new)  {Pij.shape}:\n{Pij}")

        # Correction factor for old accumulators
        cf = torch.exp(mi - mij)
        print(f"    correction factor = exp(mi_old - mi_new): {cf}")

        # Update running log-sum-exp denominator
        li_old = li.clone()
        li = cf * li + Pij.sum(dim=1)
        print(f"    li: {li_old} → {li}  (cf*li_old + rowsum(Pij))")

        # Update running output
        Oi_old_norm = Oi.norm(dim=1)
        Oi = cf.unsqueeze(1) * Oi + Pij @ Vj
        print(f"    Oi row norms: {Oi_old_norm} → {Oi.norm(dim=1)}  (cf*Oi_old + Pij@Vj)")

        # Update max
        mi = mij
        print()

    # Final output: normalize by li
    Oi = Oi / li.unsqueeze(1)
    Li = mi + torch.log(li)

    print(f"  FINAL for block i={i}:")
    print(f"    Oi (normalized) = Oi / li:\n{Oi}")
    print(f"    Li (logsumexp) = mi + log(li): {Li}\n")

    # Write back O and L
    O[i * Bq : (i + 1) * Bq, :] = Oi
    L[i * Bq : (i + 1) * Bq] = Li

print(f"\n{'='*70}")
print(f"FINAL OUTPUT")
print(f"{'='*70}")
print(f"O {O.shape}:\n{O}")
print(f"\nL {L.shape}:\n{L}")

OUTER LOOP: Q block i=0  (rows 0:2 of Q)
  Qi shape: torch.Size([2, 8])
  Qi:
tensor([[9., 2., 2., 8., 0., 0., 3., 8.],
        [6., 0., 5., 3., 7., 7., 7., 3.]])

  Init → mi: tensor([-inf, -inf])  |  li: tensor([0., 0.])
  Init → Oi shape: torch.Size([2, 8]) (all zeros)

  --- Inner loop: K block j=0  (rows 0:2 of K) ---
    Kj torch.Size([2, 8]):
tensor([[5., 8., 8., 8., 2., 3., 7., 5.],
        [0., 3., 1., 0., 3., 9., 1., 9.]])
    Vj torch.Size([2, 8]):
tensor([[5., 1., 7., 7., 6., 2., 1., 2.],
        [1., 3., 4., 4., 6., 5., 4., 9.]])

    Sij = Qi @ Kj^T / sqrt(d)  torch.Size([2, 2]):
tensor([[71.4178, 29.3449],
        [68.2358, 43.4871]])
    rowmax(Sij): tensor([71.4178, 68.2358])
    mi_old: tensor([-inf, -inf])  →  mi_new = max(mi_old, rowmax): tensor([71.4178, 68.2358])
    Pij = exp(Sij - mi_new)  torch.Size([2, 2]):
tensor([[1.0000e+00, 5.3455e-19],
        [1.0000e+00, 1.7855e-11]])
    correction factor = exp(mi_old - mi_new): tensor([0., 0.])
    li: tensor([0., 0.]

In [ ]:
import torch

In [ ]:
class NaiveFlashAttention2(torch.autograd.Function):
    @staticmethod
    def forward(ctx, Q, K, V, is_causal=False):
        # Q: (B, Nq, d)   K: (B, Nk, d)   V: (B, Nk, d)
        B, Nq, d = Q.shape
        _, Nk, _ = K.shape

        Bq = 16  # query tile size
        Bk = 16  # key tile size
        scale = 1.0 / (d ** 0.5)

        Tq = Nq // Bq
        Tk = Nk // Bk

        O = torch.zeros_like(Q)           # (B, Nq, d)
        L = torch.zeros(B, Nq, device=Q.device, dtype=Q.dtype)  # (B, Nq)

        for i in range(Tq):
            q_s = i * Bq
            q_e = q_s + Bq

            Qi = Q[:, q_s:q_e, :]            # (B, Bq, d)
            Oi = torch.zeros_like(Qi)         # (B, Bq, d)
            li = torch.zeros(B, Bq, device=Q.device, dtype=Q.dtype)  # (B, Bq)
            mi = torch.full((B, Bq), -float('inf'), device=Q.device, dtype=Q.dtype)

            for j in range(Tk):
                k_s = j * Bk
                k_e = k_s + Bk

                Kj = K[:, k_s:k_e, :]        # (B, Bk, d)
                Vj = V[:, k_s:k_e, :]        # (B, Bk, d)

                # Scores: (B, Bq, Bk)
                Sij = torch.bmm(Qi, Kj.transpose(1, 2)) * scale

                # New row-wise max: (B, Bq)
                mi_new = torch.max(mi, Sij.max(dim=2).values)

                # Exponentiate with new max: (B, Bq, Bk)
                Pij = torch.exp(Sij - mi_new.unsqueeze(2))

                # Correction factor for old accumulators: (B, Bq)
                alpha = torch.exp(mi - mi_new)

                # Update running sum and output
                li = alpha * li + Pij.sum(dim=2)
                Oi = alpha.unsqueeze(2) * Oi + torch.bmm(Pij, Vj)

                mi = mi_new

            # Normalize
            Oi = Oi / li.unsqueeze(2)
            Li = mi + torch.log(li)

            O[:, q_s:q_e, :] = Oi
            L[:, q_s:q_e] = Li

        ctx.save_for_backward(L, Q, K, V, O)
        return O

    @staticmethod
    def backward(ctx, dO):
        raise NotImplementedError


# convenience wrapper
def naive_flash_attention2(Q, K, V, is_causal=False):
    return NaiveFlashAttention2.apply(Q, K, V, is_causal)

In [ ]:
# Verify against standard attention (Eq 4-6)
B_test, Nq_test, d_test = 2, 64, 32
Q_test = torch.randn(B_test, Nq_test, d_test)
K_test = torch.randn(B_test, Nq_test, d_test)
V_test = torch.randn(B_test, Nq_test, d_test)

# Standard attention: O = softmax(Q K^T / sqrt(d)) V
scores = torch.bmm(Q_test, K_test.transpose(1, 2)) / (d_test ** 0.5)
O_ref = torch.bmm(torch.softmax(scores, dim=-1), V_test)

# Our FlashAttention-2
O_flash = naive_flash_attention2(Q_test, K_test, V_test)

print(f"Max absolute error: {(O_flash - O_ref).abs().max().item():.2e}")
print(f"All close (atol=1e-5): {torch.allclose(O_flash, O_ref, atol=1e-5)}")

In [ ]:
@triton.jit
def flash_fwd_kernel(
    Q_ptr, K_ptr, V_ptr, #these are pointers to Q,K,V
    O_ptr, L_ptr, #ptrs to output
    stride_qb, stride_qq, stride_qd,
    stride_kb, stride_kk, stride_kd,
    stride_vb, stride_vk, stride_vd,
    stride_ob, stride_oq, stride_od,
    stride_lb, stride_lq,
    N_QUERIES, N_KEYS,
    scale,
    D: tl.constexpr,
    Q_TILE_SIZE: tl.constexpr,
    K_TILE_SIZE: tl.constexpr,
):
    query_tile_index = tl.program_id(0)
    batch_index = tl.program_id(1)
    
    Q_block_ptr = tl.make_block_ptr(
        Q_ptr + batch_index * stride_qb,
        shape=(N_QUERIES, D),
        strides=(stride_qq, stride_qd),
        offsets=(query_tile_index * Q_TILE_SIZE, 0),
        block_shape=(Q_TILE_SIZE, D),
        order=(1, 0),
    )
    
    for i in range(0, tl.cdiv(N_KEYS, K_TILE_SIZE), K_TILE_SIZE):
        K_block_ptr =